In [ ]:
!git clone https://github.com/Jun1801/CoDraft-Bench.git

In [ ]:
%cd "YOUR_WORKING_DIR"
CD_PATH = "YOUR WORKING DIR"

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
import sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if CD_PATH not in sys.path:
    sys.path.append(CD_PATH)

In [ ]:
from copy import deepcopy
import numpy as np
import pandas as pd

from config import *
from preprocess import *
from model import *
from model.models import *
from utils import *


In [ ]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
import torch

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
%cd ..

In [ ]:
INPUT_ROOT = "dataset"
WORK_DIR = "working"
MODEL_NAME = "BAAI/bge-reranker-v2-m3"
MODEL_TYPE = "multi_task"
CHECKPOINT_FILE = f"{WORK_DIR}/{MODEL_NAME}_best_model.pth"
PRETRAIN_FILE = None

In [ ]:
tokenizer = get_tokenizer(MODEL_NAME)

In [ ]:
data_manager = DataManager(
    input_root = INPUT_ROOT, 
    work_dir = WORK_DIR, 
    config_data = CONFIG_DATA,
    tokenizer=tokenizer,
    seed_worker = seed_worker,
    data_generator = data_generator,
    random_seed = RANDOM_SEED
)

In [ ]:
train_ds, val_ds, test_ds = data_manager.get_dataset()
df_train, df_val, df_test = data_manager.get_data()

In [ ]:
model, _ = get_model(model_type="multi_task", model_name=MODEL_NAME, 
                  num_classes=5, num_product_classes=45,
                 device=DEVICE, **CONFIG_MODEL.MODEL_CONFIG['multi_task']['loss_args'])

In [ ]:
training_args = get_training_args(**CONFIG_MODEL.MODEL_CONFIG['multi_task']['training_args'])

In [ ]:
from transformers import Trainer, DataCollatorWithPadding
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

print("Training...")
trainer.train()

In [ ]:
test_preds, test_true = get_preds_multi(trainer, test_ds, df_test)

In [ ]:
result_df = pd.DataFrame({
    "label": test_true,
    "pred": test_preds,
})

In [ ]:
get_stats(result_df)

In [ ]:
result_df.to_csv("result_df.csv", index=False)

In [ ]:
save_model(trainer, tokenizer, MODEL_NAME, './model/checkpoint')

In [ ]:
zip_model_folder("./model")